# 05 · Events and conditional probability

Probabilities estimated from the analytic cohort: marginal, joint and conditional probabilities, independence and Bayes' rule, for events defined by fixed rules.

**Input:** `analysis_table.csv` (rows with `in_cohort`)  
**Output:** figures `05_*`

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

def find_project_root():
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "notebooks").is_dir() and (folder / "data" / "processed" / "analysis_table.csv").is_file():
            return folder.resolve()
    raise FileNotFoundError("Project root not found (needs notebooks/ and data/processed/analysis_table.csv; run notebook 04 first).")

ROOT = find_project_root()
PROC, FIG = ROOT / "data" / "processed", ROOT / "figures" / "phase2"
FIG.mkdir(parents=True, exist_ok=True)

# One colour per meaning in every figure: blue = clinical / positive, orange = molecular / negative, grey = neutral.
POS, NEG, GREY = "#2F6DB5", "#D9822B", "#6B7280"
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.titlesize": 11, "axes.labelsize": 10, "font.size": 10,
})

def save_fig(fig, name):
    path = FIG / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print("saved", path.relative_to(ROOT))

df = pd.read_csv(PROC / "analysis_table.csv")
co = df[df["in_cohort"]].copy().reset_index(drop=True)
PROLIF = "prolif_score"
print("Analytic cohort:", len(co), "patients")

## 1. Unit and independence rule

The unit is the patient. Notebook 01 keeps one RNA file per patient by a fixed rule (smallest sample ID, then smallest aliquot ID, then file ID), so no patient or sample is counted twice. Patients are treated as independent observations. Dependence through a shared hospital or batch is possible and is not modelled.

In [ ]:
assert co["patient_submitter_id"].is_unique and co["sample_submitter_id"].is_unique and co["file_id"].is_unique
print("patients:", co["patient_submitter_id"].nunique(), "| samples:", co["sample_submitter_id"].nunique(), "| RNA files:", co["file_id"].nunique())

co["site"] = co["patient_submitter_id"].str.split("-").str[1]   # tissue source site code in the TCGA barcode
sites = co["site"].value_counts()
print(f"Tissue source sites: {len(sites)} (patients per site: median {sites.median():.0f}, largest {sites.max()})")

## 2. Events, groups and cut-offs

Each patient is one observation of the random variables: proliferation score *X*, luminal score *L* and ERBB2 expression *E*.

| Event or group | Rule |
|---|---|
| ER-negative / ER-positive, PR-negative / PR-positive | IHC status |
| HER2-positive / HER2-negative | FISH, else IHC (notebook 02) |
| Triple-negative | ER-negative, PR-negative and HER2-negative |
| High proliferation | *X* above the cohort median |
| Low luminal / High luminal | *L* at or below / above the cohort median |
| Low *ERBB2* / High *ERBB2* | *E* at or below / above the cohort median |

The cut-offs are the cohort medians, fixed once and used for all conditional probabilities below.

In [ ]:
cut = {v: co[v].median() for v in [PROLIF, "luminal_score", "ERBB2"]}
print({k: round(v, 3) for k, v in cut.items()})

M = {   # one boolean mask per event / group
    "ER-negative": co["ER"].eq("Negative"), "ER-positive": co["ER"].eq("Positive"),
    "PR-negative": co["PR"].eq("Negative"), "PR-positive": co["PR"].eq("Positive"),
    "HER2-positive": co["HER2"].eq("Positive"), "HER2-negative": co["HER2"].eq("Negative"),
    "High proliferation": co[PROLIF] > cut[PROLIF], "Low proliferation": co[PROLIF] <= cut[PROLIF],
    "Low luminal": co["luminal_score"] <= cut["luminal_score"], "High luminal": co["luminal_score"] > cut["luminal_score"],
    "Low ERBB2": co["ERBB2"] <= cut["ERBB2"], "High ERBB2": co["ERBB2"] > cut["ERBB2"],
}
M["Triple-negative"] = M["ER-negative"] & M["PR-negative"] & M["HER2-negative"]
H = "High proliferation"

for a, b in [("ER-negative", "ER-positive"), ("PR-negative", "PR-positive"), ("HER2-positive", "HER2-negative"),
             (H, "Low proliferation"), ("Low luminal", "High luminal"), ("Low ERBB2", "High ERBB2")]:
    assert (M[a] ^ M[b]).all()     # complementary events cover every patient exactly once

N = len(co)
marg = pd.DataFrame({"patients": {k: int(v.sum()) for k, v in M.items()}})
marg["P(event)"] = (marg["patients"] / N).round(3)
display(marg)

## 3. Marginal and joint probabilities

For two events *A* and *B*, the 2 × 2 table gives the joint probabilities *P(A ∩ B)*; the margins are the marginal probabilities. Here: ER-negative and high proliferation.

In [ ]:
def joint_table(a, b):
    A, B = M[a], M[b]
    cnt = pd.DataFrame({b: [(A & B).sum(), (~A & B).sum()], f"not {b}": [(A & ~B).sum(), (~A & ~B).sum()]},
                       index=[a, f"not {a}"])
    cnt["total"] = cnt.sum(axis=1)
    cnt.loc["total"] = cnt.sum()
    return cnt.astype(int), (cnt / len(A)).round(3)

cnt, prob = joint_table("ER-negative", H)
display(cnt); display(prob)

Of the 940 patients, 18.3% are both ER-negative and high-proliferation, and 4.3% are ER-negative with low proliferation. Among the 212 ER-negative patients, 172 (81%) have high proliferation, against 50% overall.

## 4. Conditional probabilities

*P(A | B) = P(A ∩ B) / P(B)* is the probability of *A* among the patients in group *B*. The ratio to *P(A)* shows how conditioning changes the probability.

**Molecular group → clinical event.** Probability of ER-negative, HER2-positive and triple-negative status within the luminal and *ERBB2* groups.

In [ ]:
def cond_row(event, label, mask):
    n = int(mask.sum()); k = int((M[event] & mask).sum())
    p = k / n if n else np.nan
    return {"event": event, "given": label, "n(given)": n, "P(event | given)": round(p, 3),
            "P(event)": round(M[event].mean(), 3), "ratio": round(p / M[event].mean(), 2) if n else np.nan}

rows = [cond_row("ER-negative", g, M[g]) for g in ["Low luminal", "High luminal"]]
rows += [cond_row("HER2-positive", g, M[g]) for g in ["High ERBB2", "Low ERBB2"]]
rows += [cond_row("Triple-negative", "Low luminal & Low ERBB2", M["Low luminal"] & M["Low ERBB2"])]
rows += [cond_row("Triple-negative", "not (Low luminal & Low ERBB2)", ~(M["Low luminal"] & M["Low ERBB2"]))]
display(pd.DataFrame(rows))

ER-negative status is far more probable in the low-luminal group (0.44) than in the high-luminal group (0.01), and HER2-positive status is more probable when *ERBB2* is high (0.30) than low (0.09). Triple-negative status has probability 0.48 in patients with both low luminal score and low *ERBB2*, against 0.03 in all others. Conditioning on the molecular group changes the probability of the clinical event by a factor between 0.06 and 2.84 relative to its marginal value.

**Clinical and molecular groups → high proliferation.** Probability of high proliferation within each clinical group and each molecular group.

In [ ]:
clinical_groups = ["ER-negative", "ER-positive", "PR-negative", "PR-positive", "HER2-positive", "HER2-negative", "Triple-negative"]
molecular_groups = ["Low luminal", "High luminal", "High ERBB2", "Low ERBB2"]
tab = pd.DataFrame([cond_row(H, g, M[g]) for g in clinical_groups + molecular_groups]).drop(columns="event")
tab["type"] = ["clinical"] * len(clinical_groups) + ["molecular"] * len(molecular_groups)
display(tab)

fig, ax = plt.subplots(figsize=(9, 4))
cols = [POS if t == "clinical" else NEG for t in tab["type"]]
ax.bar(range(len(tab)), tab["P(event | given)"], color=cols, width=0.65)
for i, (p, n) in enumerate(zip(tab["P(event | given)"], tab["n(given)"])):
    ax.text(i, p + 0.01, f"{p:.2f}\n(n={n})", ha="center", va="bottom", fontsize=8,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.75), zorder=4)
baseline = M[H].mean()
ax.axhline(baseline, color=GREY, ls="--", lw=1, alpha=0.6, zorder=1.5,
           label=f"P(High proliferation) = {baseline:.2f}")
ax.set_xticks(range(len(tab))); ax.set_xticklabels(tab["given"], rotation=35, ha="right")
ax.set_ylim(0, 1.12); ax.set_ylabel("P(High proliferation | group)"); ax.grid(axis="x", visible=False)
ax.legend(loc="upper right", fontsize=8, framealpha=0.9)
ax.set_title("Probability of high proliferation by clinical (blue) and molecular (orange) group")
save_fig(fig, "05_prob_high_prolif_by_group"); plt.show()

High proliferation is more probable in the ER-negative (0.81), PR-negative (0.73), HER2-positive (0.64) and triple-negative (0.85) groups than in their complements (0.41, 0.39, 0.47) and in the low-luminal group (0.64 vs 0.36). *ERBB2* shows the opposite direction (0.45 when high, 0.55 when low). The overall probability is 0.50 by construction of the median cut-off.

**Conditioning on both.** Within the same clinical group, does the molecular group change the probability of high proliferation? Cells with few patients are unstable and shown with their size.

In [ ]:
def two_way(clin_pair, mol_pair):
    rows = []
    for c in clin_pair:
        for m in mol_pair:
            g = M[c] & M[m]; n = int(g.sum())
            rows.append({"clinical": c, "molecular": m, "n": n,
                         "P(High proliferation | both)": round(M[H][g].mean(), 3) if n else np.nan})
    return pd.DataFrame(rows)

designs = [(["ER-negative", "ER-positive"], ["Low luminal", "High luminal"]),
           (["HER2-negative", "HER2-positive"], ["Low ERBB2", "High ERBB2"])]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, (cp, mp) in zip(axes, designs):
    t = two_way(cp, mp); display(t)
    w = 0.38
    for j, m in enumerate(mp):
        s = t[t["molecular"] == m]
        x = np.arange(len(cp)) + (j - 0.5) * w
        ax.bar(x, s["P(High proliferation | both)"], width=w, color=[GREY, NEG][j], label=m)
        for xi, p, n in zip(x, s["P(High proliferation | both)"], s["n"]):
            ax.text(xi, (0 if np.isnan(p) else p) + 0.01, f"{p:.2f}\n(n={n})", ha="center", va="bottom", fontsize=8)
    ax.set_xticks(range(len(cp))); ax.set_xticklabels(cp); ax.set_ylim(0, 1.15); ax.legend(title="Molecular group", fontsize=8)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("P(High proliferation | both)")
fig.suptitle("Probability of high proliferation within clinical group, by molecular group", y=1.02)
fig.tight_layout(); save_fig(fig, "05_prob_high_prolif_two_way"); plt.show()

Within ER-positive patients, the probability of high proliferation is 0.49 in the low-luminal group (n = 264) and 0.36 in the high-luminal group (n = 464), so the luminal group shifts the probability within a clinical group. The ER-negative and high-luminal cell has only 6 patients and is not interpreted. Within HER2-negative patients, high *ERBB2* lowers the probability (0.56 to 0.35); within HER2-positive patients it raises it (0.54 to 0.67, with 41 patients in the low-*ERBB2* cell).

## 5. Independence

Events *A* and *B* are independent if *P(A ∩ B) = P(A) P(B)*, equivalently *P(A | B) = P(A)*. The table compares the observed joint probability with the product of the marginals. A ratio near 1 is consistent with independence; a ratio far from 1 shows dependence in this cohort.

In [ ]:
pairs = [("ER-negative", "Low luminal"), ("HER2-positive", "High ERBB2"),
         ("ER-negative", H), ("HER2-positive", H), ("Low luminal", H), ("High ERBB2", H)]
rows = []
for a, b in pairs:
    pa, pb, pab = M[a].mean(), M[b].mean(), (M[a] & M[b]).mean()
    rows.append({"A": a, "B": b, "P(A)": pa, "P(B)": pb, "P(A∩B)": pab, "P(A)P(B)": pa * pb,
                 "ratio": pab / (pa * pb), "P(A|B) - P(A)": pab / pb - pa})
display(pd.DataFrame(rows).round(3))

No pair is close to independence (ratios from 0.89 to 1.94). ER-negative with low luminal, and HER2-positive with high *ERBB2*, are the most dependent pairs, which is expected because the luminal score contains *ESR1* and *PGR* and *ERBB2* is the HER2 gene. High proliferation is positively dependent on all groups except high *ERBB2* (ratio 0.89). These statements describe this cohort.

## 6. Bayes' rule

The law of total probability gives *P(H) = P(H | A) P(A) + P(H | not A) P(not A)*, and Bayes' rule reverses the conditioning:

*P(A | H) = P(H | A) P(A) / P(H)*

Here *H* is the evidence (high or low proliferation) and *A* a clinical event. The rule is checked against the direct count.

In [ ]:
def bayes(event, evidence):
    A, E = M[event], M[evidence]
    prior, lik, lik_not = A.mean(), E[A].mean(), E[~A].mean()
    total = lik * prior + lik_not * (1 - prior)
    post, direct = lik * prior / total, A[E].mean()
    assert np.isclose(total, E.mean()) and np.isclose(post, direct)
    return {"event A": event, "evidence": evidence, "P(A)": prior, "P(E|A)": lik, "P(E|not A)": lik_not,
            "P(E) total prob.": total, "P(A|E) Bayes": post, "P(A|E) direct": direct}

display(pd.DataFrame([bayes(a, e) for a in ["ER-negative", "HER2-positive", "Triple-negative"]
                      for e in [H, "Low proliferation"]]).round(3))

Given high proliferation, the probability of ER-negative status rises from 0.23 to 0.37 (HER2-positive: 0.19 to 0.25; triple-negative: 0.17 to 0.29); given low proliferation it falls to 0.09 (0.14 and 0.05). Bayes' rule reproduces the direct counts. The posterior probabilities depend on the prevalences in this cohort and would change in a population with different receptor frequencies.